In [25]:
import pandas as pd
print("it works")

it works


In [26]:
import pandas as pd

df = pd.read_csv('fraudTrain.csv')
print(df.shape)
df.head()

(1296675, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [27]:
df['is_fraud'].value_counts()
df['is_fraud'].value_counts(normalize=True) * 100

is_fraud
0    99.421135
1     0.578865
Name: proportion, dtype: float64

In [28]:
df.groupby('is_fraud')['amt'].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,67.667110,154.007971,1.00,9.6100,47.280,82.540,28948.90
1,7506.0,531.320092,390.560070,1.06,245.6625,396.505,900.875,1376.04


In [29]:
df.groupby('category')['is_fraud'].mean().sort_values(ascending=False)

category
shopping_net      0.017561
misc_net          0.014458
grocery_pos       0.014098
shopping_pos      0.007225
gas_transport     0.004694
misc_pos          0.003139
grocery_net       0.002948
travel            0.002864
entertainment     0.002478
personal_care     0.002424
kids_pets         0.002114
food_dining       0.001651
home              0.001608
health_fitness    0.001549
Name: is_fraud, dtype: float64

In [30]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour'] = df['trans_date_trans_time'].dt.hour

df.groupby('hour')['is_fraud'].mean()

hour
0     0.014940
1     0.015349
2     0.014652
3     0.014239
4     0.001099
5     0.001423
6     0.000946
7     0.001327
8     0.001153
9     0.001114
10    0.000946
11    0.000998
12    0.001027
13    0.001225
14    0.001325
15    0.001208
16    0.001156
17    0.001192
18    0.001226
19    0.001236
20    0.000952
21    0.001129
22    0.028829
23    0.028374
Name: is_fraud, dtype: float64

In [31]:
import numpy as np

df['distance'] = np.sqrt((df['lat'] - df['merch_lat'])**2 + (df['long'] - df['merch_long'])**2)

df.groupby('is_fraud')['distance'].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,0.765652,0.284766,0.000239,0.564945,0.798321,0.977541,1.413364
1,7506.0,0.767281,0.281718,0.006658,0.567215,0.796925,0.976395,1.400695


In [32]:
# Age from date of birth
df['dob'] = pd.to_datetime(df['dob'])
df['age'] = 2019 - df['dob'].dt.year  # dataset is from 2019

# Final feature set
features = ['amt', 'category', 'hour', 'age', 'city_pop']
target = 'is_fraud'

model_df = df[features + [target]].copy()
model_df.head()

,amt,category,hour,age,city_pop,is_fraud
0,4.97,misc_net,0,31,3495,0
1,107.23,grocery_pos,0,41,149,0
2,220.11,entertainment,0,57,4154,0
3,45.00,gas_transport,0,52,1939,0
4,41.96,misc_pos,0,33,99,0


In [33]:
model_df = pd.get_dummies(model_df, columns=['category'], drop_first=True)
model_df.head()

,amt,hour,age,city_pop,is_fraud,category_food_dining,category_gas_transport,category_grocery_net,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
0,4.97,0,31,3495,0,False,False,False,False,False,False,False,True,False,False,False,False,False
1,107.23,0,41,149,0,False,False,False,True,False,False,False,False,False,False,False,False,False
2,220.11,0,57,4154,0,False,False,False,False,False,False,False,False,False,False,False,False,False
3,45.00,0,52,1939,0,False,True,False,False,False,False,False,False,False,False,False,False,False
4,41.96,0,33,99,0,False,False,False,False,False,False,False,False,True,False,False,False,False


In [ ]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=['is_fraud'])
y = model_df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(1037340, 17) (259335, 17)


In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.88      0.93    257834
           1       0.04      0.76      0.07      1501

    accuracy                           0.88    259335
   macro avg       0.52      0.82      0.50    259335
weighted avg       0.99      0.88      0.93    259335



In [37]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    257834
           1       0.86      0.85      0.86      1501

    accuracy                           1.00    259335
   macro avg       0.93      0.93      0.93    259335
weighted avg       1.00      1.00      1.00    259335

[[257630    204]
 [   220   1281]]


In [38]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False)

amt                        0.600014
hour                       0.192818
category_gas_transport     0.033041
age                        0.032333
city_pop                   0.026195
category_shopping_net      0.019610
category_grocery_pos       0.015578
category_food_dining       0.011351
category_home              0.011091
category_misc_net          0.009169
category_shopping_pos      0.008729
category_travel            0.008674
category_grocery_net       0.008177
category_misc_pos          0.008016
category_kids_pets         0.006361
category_health_fitness    0.004541
category_personal_care     0.004303
dtype: float64

In [39]:
import joblib

joblib.dump(rf, 'fraud_model.pkl')
joblib.dump(scaler, 'scaler.pkl')  # keeping this even though RF doesn't need it, in case you compare models later

# Also save the column order, since the API will need to build input in this exact structure
joblib.dump(X_train.columns.tolist(), 'model_columns.pkl')

['model_columns.pkl']